In [75]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import folium

# 데이터 불러오기

In [ ]:
df = pd.read_csv('../../Data_list/preprocessing_result/들락날락_위치_좌표_추출_복사본/부산광역시_좌표추가 복사본.csv')
df2 = pd.read_csv('부산광역시_문화시설_좌표추가.csv')

In [77]:
df.head(1)

,주소번호,이름,주소,위도,경도
0,27,다대도서관 들락날락,부산시 사하구 다대낙조2길 9,35.050427,128.9647


In [78]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 155 entries, 0 to 154
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   순번        155 non-null    int64  
 1   시도        155 non-null    object 
 2   시군구       145 non-null    object 
 3   주소        155 non-null    object 
 4   이름        155 non-null    object 
 5   연간 이용자 수  136 non-null    object 
 6   종류        155 non-null    object 
 7   위도        155 non-null    float64
 8   경도        155 non-null    float64
dtypes: float64(2), int64(1), object(6)
memory usage: 11.0+ KB


# 중복데이터 확인

- 들락날락 데이터프레임 내 중복 데이터 확인

In [79]:
# (위도, 경도) 쌍 기준으로 중복 확인
df_duplicates = df[df.duplicated(subset=['위도', '경도'], keep=False)]

print("df 내부 중복:")
print(df_duplicates)


df 내부 중복:
Empty DataFrame
Columns: [주소번호, 이름, 주소, 위도, 경도]
Index: []


- 문화시설 데이터프레임 내 중복 데이터 확인
    - 6종류의 위치가 2개씩 중복됨
    - 같은 건물 내 다른시설임을 확인했음

In [80]:
df2_duplicates = df2[df2.duplicated(subset=['위도', '경도'], keep=False)]

print("df2 내부 중복:")
print(df2_duplicates)


df2 내부 중복:
     순번     시도  시군구                                             주소  \
23   24  부산광역시  기장군  부산광역시 기장군 기장읍 차성로417번길 11  기장빛물꿈교육행복타운 2F, 3F   
75   24  부산광역시   서구                               부산광역시 서구 구덕로 225   
93    9  부산광역시   서구                               부산광역시 서구 구덕로 225   
111  18  부산광역시  수영구                             부산광역시 수영구 광안해변로219   
119   6  부산광역시  동래구                               부산광역시 동래구 문화로 80   
120   7  부산광역시   북구                           부산광역시 북구 금곡대로46번길 50   
121   8  부산광역시  사하구                       부산광역시 사하구 낙동남로 1233번길 25   
131   6  부산광역시  동래구                               부산광역시 동래구 문화로 80   
133   8  부산광역시   북구              부산광역시 북구 금곡대로 46번길 50 북구문화빙상센터 2층   
135  10  부산광역시  사하구        부산광역시 사하구 낙동남로 1233번길 25(하단동) 을숙도문화회관 내   
139  14  부산광역시  수영구                       부산광역시 수영구 광안해변로 219(광안동)   
154  10  부산광역시  NaN                        부산 기장군 기장읍 차성로 417번길 11   

              이름 연간 이용자 수      종류         위도          경도  
23         교리도서관   

- 각각의 데이터프레임의 위도와 경도를 묶고 교집합을 생성
- 중복된게 있는지 확인
    - 22개의 문화시설이 이미 들락날락으로 지정됐음을 확인

In [81]:
# df와 df2에서 (위도, 경도) 튜플 생성
df_loc = set(zip(df['위도'], df['경도']))
df2_loc = set(zip(df2['위도'], df2['경도']))

# 교집합 확인
duplicates = df_loc & df2_loc

# 결과 출력
print(f"중복된 좌표 개수: {len(duplicates)}")
print(duplicates)

# 중복 좌표에 해당하는 df의 '이름'과 '주소' 출력
for lat, lon in duplicates:
    matched_rows = df[(df['위도'] == lat) & (df['경도'] == lon)]
    for idx, row in matched_rows.iterrows():
        print(f"이름: {row['이름']}, 주소: {row['주소']}")
        print('-'*100)


중복된 좌표 개수: 22
{(35.272718107, 129.096198188), (35.172718668, 128.985289007), (35.156745057, 129.052320235), (35.249684877, 129.21730419), (35.137696705, 129.094567839), (35.05042701, 128.964700408), (35.202026169, 129.133807965), (35.221848653, 129.075579428), (35.086177215, 128.904767676), (35.178113166, 128.98981325), (35.238729629, 129.168697769), (35.180460087, 129.075610891), (35.170980017, 129.127154504), (35.102011405, 129.017613859), (35.325184674, 129.181485266), (35.075572812, 129.066494795), (35.185769123, 129.108783183), (35.18019645, 128.957409772), (35.110080577, 128.94487022), (35.207241325, 129.037798094), (35.078709425, 129.08019979), (35.169014991, 129.03964325)}
이름: 금정도서관(조성중), 주소: 부산광역시 금정구 금정도서관로 33(청룡동)
----------------------------------------------------------------------------------------------------
이름: 부산도서관 꿈뜨락 들락날락, 주소: 사상구 사상로 310번길 33(덕포동), 부산도서관 1층 꿈뜨락어린이실 
---------------------------------------------------------------------------------------------------

- 위도,경도쌍을 새로운 열로 추가
- 이미 들락날락으로 선정된 문화시설을 제거 후 열 드랍
- 23개의 열이 제거된 것을 확인
    - 중복은 22개인데 23개의 열이 제거된 이유는 위에서 언급한 같은건물 내에 2개의 문화시설이 있는 위치가 들락날락으로 지정돼서임
        - 을숙도문화회관과 사하문화원이 같은 건물인데 한 번에 제거됨.

In [82]:
# 1. df2에서 같은 (위도, 경도) 튜플 만들기
df2['location'] = list(zip(df2['위도'], df2['경도']))

# 3. df2에서 중복되지 않은 행만 남기기
df2_filtered = df2[~df2['location'].isin(df_loc)].drop(columns=['location'])

# 결과 확인
print(f"중복 제거 후 df2 크기: {df2_filtered.shape}")

중복 제거 후 df2 크기: (132, 9)


# 지도출력

In [86]:
import folium

# 부산의 중심 좌표 (위도, 경도)
center_lat = df['위도'].mean()
center_lon = df['경도'].mean()

# 지도 생성
m = folium.Map(location=[center_lat, center_lon], zoom_start=11)

# 들락날락 (파란색 마커)
for idx, row in df.iterrows():
    folium.Circle(
        location=[round(row['위도'], 6), round(row['경도'], 6)],
        radius=2,
        color='blue',
        fill=True,
        popup=row['이름'],
        fill_color='blue'
    ).add_to(m)
    
# 문화시설 (빨간색 마커)
for idx, row in df2_filtered.iterrows():
    if row.위도 is not None and row.경도 is not None:
        folium.Circle(
            location=[round(row['위도'], 6), round(row['경도'], 6)],
            radius=2,
            color='red',
            fill=True,
            popup=row['이름'],
            fill_color='red'
        ).add_to(m)

# 라벨 추가 (왼쪽 위에 범례처럼)
legend_html = '''
<div style="position: fixed; 
            bottom: 50px; left: 50px; width: 150px; height: 70px; 
            background-color: white; z-index:9999; font-size:14px;
            border:2px solid grey; padding: 10px;">
<b>범례</b><br>
<svg width="12" height="12"><circle cx="6" cy="6" r="5" fill="blue" /></svg> 들락날락<br>
<svg width="12" height="12"><circle cx="6" cy="6" r="5" fill="red" /></svg> 문화시설

</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

# 저장
m.save('부산_지도_라벨포함.html')

In [85]:
m